# Notebook 06: The Deutsch-Jozsa Algorithm

---

## Overview

The Deutsch-Jozsa algorithm is one of the first examples demonstrating that a quantum computer can solve certain problems **exponentially faster** than any deterministic classical computer. Although the problem it solves is somewhat artificial, it provides deep insight into the power of **quantum parallelism** and **interference**.

### What You Will Learn

1. The Deutsch problem (1-bit version) and its quantum solution
2. Full mathematical derivation of Deutsch's algorithm
3. Generalization to n bits: the Deutsch-Jozsa algorithm
4. Oracle construction and the Hadamard transform on $n$ qubits
5. Quantum parallelism — how superposition enables evaluating $f$ on all inputs simultaneously
6. Complete Qiskit implementations with verification

### Prerequisites

- Familiarity with single-qubit gates ($X$, $H$, $Z$) and multi-qubit gates (CNOT)
- Basic understanding of Dirac notation: $|\psi\rangle$, $\langle\psi|$
- Tensor products and measurement in the computational basis

## 1. The Problem: Constant vs. Balanced Functions

Consider a Boolean function $f: \{0,1\}^n \to \{0,1\}$. We are **promised** that $f$ is one of two types:

| Type | Definition |
|------|------------|
| **Constant** | $f(x) = 0$ for all $x$, or $f(x) = 1$ for all $x$ |
| **Balanced** | $f(x) = 0$ for exactly half the inputs, and $f(x) = 1$ for the other half |

**Goal:** Determine whether $f$ is constant or balanced.

### Classical Complexity

Classically, in the **worst case**, we need to evaluate $f$ on $2^{n-1} + 1$ inputs. If all $2^{n-1}$ evaluations return the same value, we still cannot be certain until we check one more. This is $O(2^n)$ queries.

### Quantum Advantage

The Deutsch-Jozsa algorithm solves this problem with **exactly 1 query** to $f$ — an exponential speedup!

## 2. The Deutsch Problem (1-Bit Case)

Let us start with the simplest case: $n = 1$. Here $f: \{0,1\} \to \{0,1\}$.

There are exactly four possible functions:

| Function | $f(0)$ | $f(1)$ | Type |
|----------|--------|--------|------|
| $f_1$ | 0 | 0 | Constant |
| $f_2$ | 1 | 1 | Constant |
| $f_3$ | 0 | 1 | Balanced |
| $f_4$ | 1 | 0 | Balanced |

Classically, we need **2 queries** (evaluate both $f(0)$ and $f(1)$) to distinguish constant from balanced with certainty.

Deutsch showed that a quantum computer can do it with **1 query**.

## 3. Quantum Oracles

Since quantum operations must be **unitary** (and hence reversible), we cannot simply compute $f(x)$ in place. Instead, we use an **oracle** (also called a "black box") that acts on two registers:

$$U_f |x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle$$

where $\oplus$ denotes addition modulo 2 (XOR).

### Key Properties

1. **Reversibility:** Applying $U_f$ twice gives back the original state: $U_f^2 = I$
2. **The input register $|x\rangle$ is unchanged** — only the target register is modified
3. If $|y\rangle = |0\rangle$, then $U_f|x\rangle|0\rangle = |x\rangle|f(x)\rangle$

### Phase Kickback Trick

Now, here is the crucial insight. Suppose the target qubit is in the state $|-\rangle = \frac{1}{\sqrt{2}}(|0\rangle - |1\rangle)$. Then:

$$U_f |x\rangle |-\rangle = U_f |x\rangle \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

$$= |x\rangle \frac{|0 \oplus f(x)\rangle - |1 \oplus f(x)\rangle}{\sqrt{2}}$$

**Case 1:** If $f(x) = 0$:
$$= |x\rangle \frac{|0\rangle - |1\rangle}{\sqrt{2}} = |x\rangle |-\rangle$$

**Case 2:** If $f(x) = 1$:
$$= |x\rangle \frac{|1\rangle - |0\rangle}{\sqrt{2}} = -|x\rangle |-\rangle$$

Combining both cases:

$$\boxed{U_f |x\rangle |-\rangle = (-1)^{f(x)} |x\rangle |-\rangle}$$

The function value $f(x)$ has been "kicked back" into the **phase** of the input register! The target qubit $|-\rangle$ is unchanged. This is called **phase kickback**.

## 4. Deutsch's Algorithm — Full Derivation

### Circuit

```
|0⟩ ─── H ─── ┤       ├─── H ─── Measure
               │  U_f  │
|1⟩ ─── H ─── ┤       ├──────────────────
```

### Step-by-Step Derivation

**Step 0: Initial State**

$$|\psi_0\rangle = |0\rangle|1\rangle$$

**Step 1: Apply Hadamard to both qubits**

$$|\psi_1\rangle = H|0\rangle \otimes H|1\rangle = |+\rangle|-\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}} \otimes \frac{|0\rangle - |1\rangle}{\sqrt{2}}$$

**Step 2: Apply the oracle $U_f$**

Using phase kickback:

$$|\psi_2\rangle = \frac{1}{\sqrt{2}}\left[(-1)^{f(0)}|0\rangle + (-1)^{f(1)}|1\rangle\right] |-\rangle$$

$$= \frac{(-1)^{f(0)}}{\sqrt{2}}\left[|0\rangle + (-1)^{f(0)\oplus f(1)}|1\rangle\right] |-\rangle$$

**Step 3: Apply Hadamard to the first qubit**

Recall that $H|0\rangle = |+\rangle$ and $H|1\rangle = |-\rangle$, so:
- $H|+\rangle = |0\rangle$ and $H|-\rangle = |1\rangle$

**Case A: $f(0) \oplus f(1) = 0$ (constant)**

The first qubit is $|+\rangle$ (up to global phase), so $H|+\rangle = |0\rangle$.

$$|\psi_3\rangle = \pm|0\rangle|-\rangle$$

**Case B: $f(0) \oplus f(1) = 1$ (balanced)**

The first qubit is $|-\rangle$ (up to global phase), so $H|-\rangle = |1\rangle$.

$$|\psi_3\rangle = \pm|1\rangle|-\rangle$$

**Step 4: Measure the first qubit**

$$\boxed{\text{Measure } |0\rangle \Rightarrow f \text{ is constant}, \quad \text{Measure } |1\rangle \Rightarrow f \text{ is balanced}}$$

With **one single query** to the oracle, we have determined the property of $f$!

## 5. Implementation — Deutsch's Algorithm in Qiskit

Let us now implement the 1-bit Deutsch algorithm.

In [ ]:
# Install/import dependencies
# pip install qiskit qiskit-aer matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

print("Imports successful!")

In [ ]:
def deutsch_oracle(circuit, case):
    """
    Add a Deutsch oracle to the circuit.
    
    Parameters:
    -----------
    circuit : QuantumCircuit with 2 qubits (qubit 0 = input, qubit 1 = target)
    case : str
        'constant_0' : f(x) = 0 for all x  (identity — do nothing)
        'constant_1' : f(x) = 1 for all x  (flip target unconditionally)
        'balanced_id': f(x) = x            (CNOT)
        'balanced_not': f(x) = NOT(x)       (X on input, CNOT, X on input)
    """
    if case == 'constant_0':
        # f(x) = 0: do nothing (identity)
        pass
    elif case == 'constant_1':
        # f(x) = 1: flip target regardless of input
        circuit.x(1)
    elif case == 'balanced_id':
        # f(x) = x: CNOT from input to target
        circuit.cx(0, 1)
    elif case == 'balanced_not':
        # f(x) = NOT(x): flip input, CNOT, flip input back
        circuit.x(0)
        circuit.cx(0, 1)
        circuit.x(0)
    else:
        raise ValueError(f"Unknown case: {case}")
    
    return circuit

print("Oracle function defined.")

In [ ]:
def deutsch_algorithm(oracle_case):
    """
    Implement Deutsch's algorithm for a given oracle.
    Returns the circuit and measurement result.
    """
    # Create circuit: qubit 0 = input, qubit 1 = target
    qc = QuantumCircuit(2, 1)
    
    # Step 1: Initialize |01⟩
    qc.x(1)  # qubit 1 -> |1⟩
    qc.barrier()
    
    # Step 2: Apply H to both qubits → |+⟩|−⟩
    qc.h(0)
    qc.h(1)
    qc.barrier()
    
    # Step 3: Apply oracle
    deutsch_oracle(qc, oracle_case)
    qc.barrier()
    
    # Step 4: Apply H to input qubit
    qc.h(0)
    qc.barrier()
    
    # Step 5: Measure input qubit
    qc.measure(0, 0)
    
    return qc

# Test all four cases
cases = ['constant_0', 'constant_1', 'balanced_id', 'balanced_not']
sim = AerSimulator()

print("Deutsch's Algorithm Results:")
print("=" * 50)

for case in cases:
    qc = deutsch_algorithm(case)
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()
    
    measured_bit = max(counts, key=counts.get)
    verdict = "CONSTANT" if measured_bit == '0' else "BALANCED"
    print(f"Oracle: {case:15s} → Measured: {measured_bit} → {verdict}")

print("\nAll results are deterministic (no probabilistic outcomes)!")

In [ ]:
# Visualize the circuit for the balanced_id case
qc_vis = deutsch_algorithm('balanced_id')
qc_vis.draw('mpl', style='iqp', fold=False)
plt.title("Deutsch's Algorithm — Balanced Oracle f(x)=x", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Quantum Parallelism

The power of the Deutsch-Jozsa algorithm comes from **quantum parallelism**. Let us understand this concept carefully.

### Classical Evaluation

A classical computer evaluates $f$ on **one input at a time**:
$$x_0 \to f(x_0), \quad x_1 \to f(x_1), \quad \ldots$$

### Quantum Evaluation

A quantum computer can prepare a superposition of **all inputs** and evaluate $f$ on all of them **simultaneously**:

$$U_f \left(\frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} |x\rangle\right)|y\rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} |x\rangle |y \oplus f(x)\rangle$$

### The Catch

Quantum parallelism alone is **not sufficient** for a speedup. If we simply measure the output, we get $f(x)$ for a **random** $x$ — no better than classical.

The key is to combine quantum parallelism with **interference** to extract **global properties** of $f$ (like whether it is constant or balanced) without reading individual values.

### The Interference Pattern

In the Deutsch-Jozsa algorithm:
1. The Hadamard creates a uniform superposition (quantum parallelism)
2. The oracle encodes $f(x)$ in the phases: $(-1)^{f(x)}$
3. The final Hadamard creates **constructive interference** for $|0\rangle^{\otimes n}$ if $f$ is constant, or **destructive interference** if $f$ is balanced

## 7. The Deutsch-Jozsa Algorithm — General Case ($n$ Qubits)

### Setup

We have $f: \{0,1\}^n \to \{0,1\}$, promised to be either constant or balanced.

### The Circuit

$$|0\rangle^{\otimes n}|1\rangle \xrightarrow{H^{\otimes (n+1)}} \xrightarrow{U_f} \xrightarrow{H^{\otimes n} \otimes I} \xrightarrow{\text{Measure}}$$

### Full Mathematical Derivation

**Step 0:** Initial state
$$|\psi_0\rangle = |0\rangle^{\otimes n} |1\rangle$$

**Step 1:** Apply $H^{\otimes (n+1)}$

The Hadamard transform on a computational basis state $|x\rangle$ where $x \in \{0,1\}$ is:
$$H|x\rangle = \frac{1}{\sqrt{2}}\sum_{z \in \{0,1\}} (-1)^{xz} |z\rangle$$

For $n$ qubits, $H^{\otimes n}|0\rangle^{\otimes n}$:

$$H^{\otimes n}|0\rangle^{\otimes n} = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} |x\rangle$$

And $H|1\rangle = |-\rangle$. So:

$$|\psi_1\rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} |x\rangle \otimes |-\rangle$$

**Step 2:** Apply oracle $U_f$ (with phase kickback)

$$|\psi_2\rangle = \frac{1}{\sqrt{2^n}} \sum_{x=0}^{2^n - 1} (-1)^{f(x)} |x\rangle \otimes |-\rangle$$

**Step 3:** Apply $H^{\otimes n}$ to the first $n$ qubits

Using the general Hadamard identity for an $n$-bit string $x$:

$$H^{\otimes n}|x\rangle = \frac{1}{\sqrt{2^n}} \sum_{z=0}^{2^n - 1} (-1)^{x \cdot z} |z\rangle$$

where $x \cdot z = \sum_{i=0}^{n-1} x_i z_i \pmod{2}$ is the bitwise inner product.

Therefore:

$$|\psi_3\rangle = \frac{1}{2^n} \sum_{z=0}^{2^n - 1} \left[ \sum_{x=0}^{2^n - 1} (-1)^{f(x) + x \cdot z} \right] |z\rangle \otimes |-\rangle$$

**Step 4:** Measure the first $n$ qubits

The probability of measuring $|0\rangle^{\otimes n}$ (i.e., $z = 0$):

$$P(z=0) = \left| \frac{1}{2^n} \sum_{x=0}^{2^n - 1} (-1)^{f(x)} \right|^2$$

**If $f$ is constant:** All $(-1)^{f(x)}$ terms are the same ($+1$ or $-1$):
$$P(z=0) = \left| \frac{1}{2^n} \cdot (\pm 2^n) \right|^2 = 1$$

**If $f$ is balanced:** Half the terms are $+1$ and half are $-1$:
$$P(z=0) = \left| \frac{1}{2^n} \cdot 0 \right|^2 = 0$$

$$\boxed{P(z = 0^n) = \begin{cases} 1 & \text{if } f \text{ is constant} \\ 0 & \text{if } f \text{ is balanced} \end{cases}}$$

## 8. Constructing Oracles for $n$-Qubit Deutsch-Jozsa

### Constant Oracles

- **$f(x) = 0$:** Do nothing (identity on target qubit)
- **$f(x) = 1$:** Apply $X$ gate to the target qubit

### Balanced Oracles

A balanced oracle can be constructed by applying CNOT gates from a subset of the input qubits to the target qubit. For example:

- **Inner product oracle:** $f(x) = x_0 \oplus x_1 \oplus \cdots \oplus x_{n-1}$ — CNOT from each input qubit to the target
- **Single-bit oracle:** $f(x) = x_k$ for some $k$ — CNOT from qubit $k$ to the target
- **General balanced:** We can also add $X$ gates before some CNOTs to negate certain bits

In [ ]:
def dj_oracle(n, oracle_type, custom_bits=None):
    """
    Create a Deutsch-Jozsa oracle for n input qubits + 1 target qubit.
    
    Parameters:
    -----------
    n : int — number of input qubits
    oracle_type : str — 'constant_0', 'constant_1', 'balanced_inner', 
                        'balanced_single_k', or 'balanced_custom'
    custom_bits : list — for 'balanced_custom', which input qubits to CNOT.
                         For 'balanced_single_k', which qubit index k.
    
    Returns: QuantumCircuit acting on n+1 qubits
    """
    oracle_qc = QuantumCircuit(n + 1, name='Oracle')
    target = n  # last qubit is the target
    
    if oracle_type == 'constant_0':
        # f(x) = 0: identity
        pass
    
    elif oracle_type == 'constant_1':
        # f(x) = 1: flip target
        oracle_qc.x(target)
    
    elif oracle_type == 'balanced_inner':
        # f(x) = x_0 ⊕ x_1 ⊕ ... ⊕ x_{n-1}
        for i in range(n):
            oracle_qc.cx(i, target)
    
    elif oracle_type == 'balanced_single_k':
        # f(x) = x_k
        k = custom_bits if isinstance(custom_bits, int) else custom_bits[0]
        oracle_qc.cx(k, target)
    
    elif oracle_type == 'balanced_custom':
        # f(x) = XOR of selected bits, with optional NOT on some
        for i in custom_bits:
            oracle_qc.cx(i, target)
    
    return oracle_qc.to_gate()

print("DJ oracle function defined.")

In [ ]:
def deutsch_jozsa(n, oracle_type, custom_bits=None):
    """
    Full Deutsch-Jozsa algorithm for n input qubits.
    
    Returns: QuantumCircuit, measurement counts
    """
    # n input qubits + 1 target qubit, n classical bits for measurement
    qc = QuantumCircuit(n + 1, n)
    
    # Step 1: Initialize target qubit to |1⟩
    qc.x(n)
    qc.barrier()
    
    # Step 2: Apply H to all qubits
    for i in range(n + 1):
        qc.h(i)
    qc.barrier()
    
    # Step 3: Apply oracle
    oracle_gate = dj_oracle(n, oracle_type, custom_bits)
    qc.append(oracle_gate, range(n + 1))
    qc.barrier()
    
    # Step 4: Apply H to input qubits only
    for i in range(n):
        qc.h(i)
    qc.barrier()
    
    # Step 5: Measure input qubits
    for i in range(n):
        qc.measure(i, i)
    
    # Simulate
    sim = AerSimulator()
    compiled = transpile(qc, sim)
    result = sim.run(compiled, shots=1024).result()
    counts = result.get_counts()
    
    return qc, counts

print("Deutsch-Jozsa algorithm function defined.")

In [ ]:
# Test with n=3 qubits — constant oracle f(x) = 0
n = 3
qc_const, counts_const = deutsch_jozsa(n, 'constant_0')

print(f"Deutsch-Jozsa with n={n}, oracle = constant_0")
print(f"Measurement results: {counts_const}")
result_str = max(counts_const, key=counts_const.get)
print(f"Most likely outcome: |{result_str}⟩")
print(f"Verdict: {'CONSTANT' if result_str == '0' * n else 'BALANCED'}")
print()

# Draw the circuit
qc_const.draw('mpl', style='iqp', fold=False)
plt.title(f"Deutsch-Jozsa (n={n}): Constant Oracle f(x)=0", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Test with n=3 qubits — balanced oracle f(x) = x_0 ⊕ x_1 ⊕ x_2
qc_bal, counts_bal = deutsch_jozsa(n, 'balanced_inner')

print(f"Deutsch-Jozsa with n={n}, oracle = balanced (inner product)")
print(f"Measurement results: {counts_bal}")
result_str = max(counts_bal, key=counts_bal.get)
print(f"Most likely outcome: |{result_str}⟩")
print(f"Verdict: {'CONSTANT' if result_str == '0' * n else 'BALANCED'}")

In [ ]:
# Comprehensive test: all oracle types for n=4
n = 4

test_cases = [
    ('constant_0', None, 'Constant'),
    ('constant_1', None, 'Constant'),
    ('balanced_inner', None, 'Balanced'),
    ('balanced_single_k', 0, 'Balanced'),
    ('balanced_single_k', 2, 'Balanced'),
    ('balanced_custom', [0, 1], 'Balanced'),
    ('balanced_custom', [1, 2, 3], 'Balanced'),
]

print(f"Deutsch-Jozsa Algorithm — Comprehensive Test (n={n})")
print("=" * 65)
print(f"{'Oracle Type':<25} {'Params':<12} {'Result':<10} {'Correct?'}")
print("-" * 65)

for oracle_type, params, expected in test_cases:
    _, counts = deutsch_jozsa(n, oracle_type, params)
    result_str = max(counts, key=counts.get)
    verdict = 'Constant' if result_str == '0' * n else 'Balanced'
    correct = '✓' if verdict == expected else '✗'
    param_str = str(params) if params is not None else '-'
    print(f"{oracle_type:<25} {param_str:<12} {verdict:<10} {correct}")

print("\nAll tests passed — the algorithm correctly identifies every oracle!")

## 9. Visualizing the Statevector

Let us peek inside the quantum state at each stage to see the interference in action.

In [ ]:
from qiskit.quantum_info import Statevector

def visualize_dj_stages(n, oracle_type, custom_bits=None):
    """
    Show the statevector at each stage of the Deutsch-Jozsa algorithm.
    We only track the input register amplitudes (trace out the target).
    """
    # Build circuit step by step (no measurement for statevector sim)
    qc = QuantumCircuit(n + 1)
    
    # Stage 0: |0...0⟩|1⟩
    qc.x(n)
    sv0 = Statevector.from_instruction(qc)
    
    # Stage 1: Apply H to all
    for i in range(n + 1):
        qc.h(i)
    sv1 = Statevector.from_instruction(qc)
    
    # Stage 2: Apply oracle
    oracle_gate = dj_oracle(n, oracle_type, custom_bits)
    qc.append(oracle_gate, range(n + 1))
    sv2 = Statevector.from_instruction(qc)
    
    # Stage 3: Apply H to input qubits
    for i in range(n):
        qc.h(i)
    sv3 = Statevector.from_instruction(qc)
    
    # Extract amplitudes of the input register
    # (the target is always |−⟩, so we look at states where target=1
    #  and scale by √2 to normalize the input register)
    
    stages = {
        'After H (superposition)': sv1,
        'After Oracle (phase encoded)': sv2,
        'After final H (interference)': sv3
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    
    for idx, (title, sv) in enumerate(stages.items()):
        # Get probabilities for the full system
        probs = np.abs(sv.data)**2
        
        # Sum over target qubit to get input register probabilities
        input_probs = np.zeros(2**n)
        for state_idx in range(2**(n+1)):
            input_idx = state_idx >> 1  # remove target qubit (LSB in Qiskit)
            input_probs[input_idx] += probs[state_idx]
        
        labels = [format(i, f'0{n}b') for i in range(2**n)]
        axes[idx].bar(labels, input_probs, color='steelblue', alpha=0.8)
        axes[idx].set_title(title, fontsize=11)
        axes[idx].set_ylabel('Probability')
        axes[idx].set_ylim(0, 1.1)
        if n > 3:
            axes[idx].tick_params(axis='x', rotation=90, labelsize=7)
    
    plt.suptitle(f'Deutsch-Jozsa Statevector Evolution (n={n}, {oracle_type})', fontsize=14)
    plt.tight_layout()
    plt.show()

# Visualize constant oracle
visualize_dj_stages(3, 'constant_0')

# Visualize balanced oracle
visualize_dj_stages(3, 'balanced_inner')

## 10. Understanding the Hadamard Transform on $n$ Qubits

The $n$-qubit Hadamard transform $H^{\otimes n}$ plays a central role. Let us examine its matrix structure.

### Definition

$$H^{\otimes n} = \underbrace{H \otimes H \otimes \cdots \otimes H}_{n \text{ times}}$$

### Matrix Elements

The $(z, x)$-th element of the $2^n \times 2^n$ matrix $H^{\otimes n}$ is:

$$(H^{\otimes n})_{z,x} = \frac{1}{\sqrt{2^n}} (-1)^{x \cdot z}$$

### Action on Basis States

$$H^{\otimes n} |x\rangle = \frac{1}{\sqrt{2^n}} \sum_{z=0}^{2^n - 1} (-1)^{x \cdot z} |z\rangle$$

This is essentially a **discrete Fourier transform over $\mathbb{Z}_2^n$** (the Walsh-Hadamard transform).

### Example: $n = 2$

$$H^{\otimes 2} = \frac{1}{2} \begin{pmatrix} 1 & 1 & 1 & 1 \\ 1 & -1 & 1 & -1 \\ 1 & 1 & -1 & -1 \\ 1 & -1 & -1 & 1 \end{pmatrix}$$

In [ ]:
from qiskit.quantum_info import Operator

# Visualize the Hadamard matrix for n=1,2,3
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for idx, n in enumerate([1, 2, 3]):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
    
    op = Operator(qc)
    matrix = np.real(op.data * np.sqrt(2**n))  # Scale to show ±1 pattern
    
    im = axes[idx].imshow(matrix, cmap='RdBu', vmin=-1.5, vmax=1.5)
    axes[idx].set_title(f'$H^{{\\otimes {n}}}$ (scaled by $\\sqrt{{{2**n}}}$)', fontsize=12)
    axes[idx].set_xlabel('Column (x)')
    axes[idx].set_ylabel('Row (z)')
    
    # Add text annotations for small matrices
    if n <= 2:
        for i in range(2**n):
            for j in range(2**n):
                axes[idx].text(j, i, f'{matrix[i,j]:+.0f}', ha='center', va='center', fontsize=10)

plt.colorbar(im, ax=axes, shrink=0.8, label='Value')
plt.suptitle('Hadamard Transform Matrices (showing ±1 pattern)', fontsize=14)
plt.tight_layout()
plt.show()

## 11. Scaling Analysis

Let us verify that the Deutsch-Jozsa algorithm works correctly for increasing $n$ and compare the quantum query complexity with the classical one.

In [ ]:
# Test DJ for increasing n
import time

print("Deutsch-Jozsa Scaling Test")
print("=" * 60)
print(f"{'n':<5} {'N=2^n':<10} {'Classical Queries':<20} {'Quantum Queries':<15} {'Result'}")
print("-" * 60)

for n in range(1, 9):
    N = 2**n
    classical_queries = N // 2 + 1  # worst case
    quantum_queries = 1  # always 1!
    
    _, counts = deutsch_jozsa(n, 'balanced_inner')
    result_str = max(counts, key=counts.get)
    verdict = 'Constant' if result_str == '0' * n else 'Balanced'
    
    print(f"{n:<5} {N:<10} {classical_queries:<20} {quantum_queries:<15} {verdict}")

print("\nQuantum complexity: O(1) queries")
print("Classical complexity: O(2^n) queries")
print("Speedup: EXPONENTIAL!")

In [ ]:
# Visualize the exponential gap
ns = list(range(1, 16))
classical = [2**(n-1) + 1 for n in ns]
quantum = [1 for _ in ns]

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(ns, classical, 'ro-', linewidth=2, markersize=8, label='Classical (worst case)')
ax.semilogy(ns, quantum, 'bs-', linewidth=2, markersize=8, label='Quantum (Deutsch-Jozsa)')

ax.set_xlabel('Number of input bits (n)', fontsize=13)
ax.set_ylabel('Number of oracle queries', fontsize=13)
ax.set_title('Deutsch-Jozsa: Quantum vs Classical Query Complexity', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(ns)

# Annotate the gap
ax.annotate('Exponential gap!', xy=(10, 500), fontsize=14, color='red',
            arrowprops=dict(arrowstyle='->', color='red'),
            xytext=(7, 5000))

plt.tight_layout()
plt.show()

## 12. Limitations and Context

### What Deutsch-Jozsa Does NOT Show

1. **The problem is artificial**: The "constant vs. balanced" promise is not a problem people typically need to solve.

2. **Probabilistic classical algorithms close the gap**: A randomized classical algorithm can solve this with $O(1)$ queries (high probability) — just sample a few random inputs. If you see both 0 and 1, it is balanced. After $k$ samples, the failure probability is $2^{-k}$.

3. **The exponential speedup is only over *deterministic* classical algorithms.**

### What Deutsch-Jozsa DOES Show

1. **Quantum parallelism is real**: A quantum computer can evaluate $f$ on all inputs in superposition.

2. **Phase kickback is a powerful technique**: Encoding function values in phases enables interference.

3. **The algorithmic structure is a template**: The same pattern — $H^{\otimes n} \to U_f \to H^{\otimes n} \to \text{Measure}$ — appears in many quantum algorithms (Bernstein-Vazirani, Simon's, etc.).

4. **It is a building block**: Understanding Deutsch-Jozsa is essential for Grover's search and Shor's factoring.

## 13. Exercises

1. **Modify the oracle** to implement $f(x_0, x_1, x_2) = x_0 \oplus x_2$ (skipping $x_1$). Verify it is detected as balanced.

2. **Prove algebraically** that for a balanced function, $P(z = 0^n) = 0$ by expanding the sum $\sum_x (-1)^{f(x)}$.

3. **Bernstein-Vazirani extension**: The same circuit structure can find a secret string $s$ in $f(x) = s \cdot x \pmod{2}$ with one query. Implement this in Qiskit.

4. **Noise analysis**: Add depolarizing noise to the simulation and plot how the algorithm's success probability degrades with increasing noise and $n$.

5. **Circuit depth**: For the balanced inner-product oracle on $n$ qubits, what is the circuit depth? How does it scale with $n$?

## 14. Summary

| Concept | Details |
|---------|--------|
| **Problem** | Determine if $f:\{0,1\}^n \to \{0,1\}$ is constant or balanced |
| **Classical** | $O(2^n)$ deterministic queries |
| **Quantum** | $O(1)$ queries — exactly **one** |
| **Key techniques** | Phase kickback, quantum parallelism, interference |
| **Algorithm structure** | $H^{\otimes n} \to U_f \to H^{\otimes n} \to \text{Measure}$ |
| **Measurement rule** | $|0^n\rangle \Rightarrow$ constant, anything else $\Rightarrow$ balanced |
| **Limitation** | Probabilistic classical can also solve in $O(1)$ queries (with high probability) |

**Next:** [Notebook 07 — Grover's Search Algorithm](07-grovers-search-algorithm.ipynb)